In [ ]:
import re
import pandas as pd

# 1. Leer el archivo como texto plano
with open('llegadas_2019_2018_clean.csv', 'r', encoding='utf-8-sig', errors='ignore') as f:
    lineas = f.readlines()

lineas_datos = [l.strip() for l in lineas if l.strip() != ""]
if "RO" in lineas_datos[0]:
    lineas_datos = lineas_datos[1:]

# 2. ORDEN REAL DE LAS COLUMNAS (Tal como viene en tu archivo, de más nuevo a más viejo)
meses_columnas = [
    "2019-12", "2019-11", "2019-10", "2019-09", "2019-08", "2019-07",
    "2019-06", "2019-05", "2019-04", "2019-03", "2019-02", "2019-01",
    "2018-12", "2018-11", "2018-10", "2018-09", "2018-08", "2018-07",
    "2018-06", "2018-05", "2018-04", "2018-03", "2018-02", "2018-01"
]

matriz_paises = {}

# 3. Extraer la serie temporal de cada país
for linea in lineas_datos[1:]:
    if "Pasajeros Totales" in linea:
        continue

    paises_encontrados = re.findall(r'[A-ZÁÉÍÓÚÑüÜ]{3,}(?:\s+[A-ZÁÉÍÓÚÑüÜ]+)*', linea)
    if not paises_encontrados:
        continue
    pais = paises_encontrados[0].strip()

    numeros = re.findall(r'"([\d\.]+)"', linea)
    if not numeros:
        numeros = re.findall(r'\b\d+(?:\.\d+)*\b', linea)

    valores_meses = []
    for num in numeros:
        # Reemplazamos los puntos de miles.
        # NOTA: Si un número viene como '13.58' (sin el cero), lo tratamos con cuidado.
        num_limpio = num.replace('.', '')
        if num_limpio.isdigit():
            valores_meses.append(int(num_limpio))

    matriz_paises[pais] = valores_meses

# 4. Construir la estructura temporal mes a mes
paises_top = {'ESPAÑA', 'SPAIN', 'FRANCIA', 'FRANCE', 'ITALIA', 'ITALY',
              'REINO UNIDO', 'UNITED KINGDOM', 'UK', 'PAISES BAJOS', 'HOLANDA',
              'NETHERLANDS', 'ALEMANIA', 'GERMANY'}

datos_historicos_finales = []

for idx_mes, mes_nombre in enumerate(meses_columnas):

    def extraer_valor_mes(nombre_pais):
        for k, lista_valores in matriz_paises.items():
            if nombre_pais in k and idx_mes < len(lista_valores):
                return lista_valores[idx_mes]
        return 0

    nac = extraer_valor_mes('ESPAÑA') or extraer_valor_mes('SPAIN')
    fr = extraer_valor_mes('FRANCIA') or extraer_valor_mes('FRANCE')
    it = extraer_valor_mes('ITALIA') or extraer_valor_mes('ITALY')
    uk = extraer_valor_mes('REINO UNIDO') or extraer_valor_mes('UNITED KINGDOM') or extraer_valor_mes('UK')
    nl = extraer_valor_mes('PAISES BAJOS') or extraer_valor_mes('HOLANDA') or extraer_valor_mes('NETHERLANDS')
    de = extraer_valor_mes('ALEMANIA') or extraer_valor_mes('GERMANY')

    resto_del_mundo = 0
    for k, lista_valores in matriz_paises.items():
        if not any(p in k for p in paises_top) and idx_mes < len(lista_valores):
            resto_del_mundo += lista_valores[idx_mes]

    internacionales = 0
    for k, lista_valores in matriz_paises.items():
        if 'ESPAÑA' not in k and 'SPAIN' not in k and idx_mes < len(lista_valores):
            internacionales += lista_valores[idx_mes]

    total_aeropuerto = nac + internacionales

    datos_historicos_finales.append({
        'Mes': mes_nombre,
        'Volumen total llegadas aeropuerto': total_aeropuerto,
        'Volumen llegadas Nacionales': nac,
        'Volumen llegadas Internacionales': internacionales,
        'Volumen llegadas Francia': fr,
        'Volumen llegadas Italia': it,
        'Volumen llegadas Reino Unido': uk,
        'Volumen llegadas Países Bajos': nl,
        'Volumen llegadas Alemania': de,
        'Volumen llegadas Resto del Mundo': resto_del_mundo
    })

# 5. Convertir a DataFrame, ordenar cronológicamente de viejo a nuevo y guardar
df_temporal = pd.DataFrame(datos_historicos_finales)

# Lo ordenamos para que empiece en 2018-01 y termine en 2019-12 (así te será más fácil pegarlo después)
df_temporal = df_temporal.sort_values(by='Mes').reset_index(drop=True)

df_temporal.to_csv('2019-2018.csv', index=False, encoding='utf-8-sig')


¡Corregido! Guardado como '2019-2018.csv' con el orden de datos exacto.


,Mes,Volumen total llegadas aeropuerto,Volumen llegadas Nacionales,Volumen llegadas Internacionales,Volumen llegadas Francia,Volumen llegadas Italia,Volumen llegadas Reino Unido,Volumen llegadas Países Bajos,Volumen llegadas Alemania,Volumen llegadas Resto del Mundo
0,2018-01,205577,105885,99692,20298,23608,16080,7092,10793,21821
1,2018-02,217904,109310,108594,22481,22734,19973,7883,13346,22177
2,2018-03,272833,143114,129719,27389,24925,22393,9931,17482,27599
3,2018-04,281013,139045,141968,35221,26643,20791,9525,19009,30779
4,2018-05,282127,139225,142902,34490,26339,21783,10769,18082,31439
5,2018-06,267735,130172,137563,33190,25379,22954,10762,15907,29371
6,2018-07,268959,134066,134893,29553,27517,24494,8289,17010,28030
7,2018-08,261372,131209,130163,26717,27993,24009,7121,17923,26400
8,2018-09,282764,138396,144368,35291,26991,22928,11476,16972,30710
9,2018-10,300926,148308,152618,37063,29654,22952,13071,16554,33324
